# F1 Strategy Predictor — PyTorch
## Real-Time Pit Stop & Compound Prediction

---
Conversione da TensorFlow/Keras a PyTorch.
Architettura invariata: Transformer causale seq-to-seq con causal mask lower-triangolare.

| | Keras (originale) | **PyTorch (questo notebook)** |
|---|---|---|
| Framework | TensorFlow 2.x / Keras | **PyTorch** |
| Loss masking | `tf.where` + `tf.reduce_sum` | `torch.where` + `tensor.sum()` |
| Causal mask | `tf.linalg.band_part` (bool) | `torch.tril` + `-inf` additive mask |
| Dataset | `model.fit(X, y)` | `torch.utils.data.DataLoader` |
| Salvataggio | `.keras` file | `torch.save` (state_dict) |

### Requirements
- `label_encoder.pkl` (from DataAnalysis.ipynb)
- `f1_dataset_clean.pkl` (from DataAnalysis.ipynb)

# 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
if not hasattr(np, 'NaN'):
    np.NaN = np.nan
import pandas as pd
import matplotlib.pyplot as plt
import requests
import joblib
import json
import os
from io import BytesIO

os.makedirs('Model', exist_ok=True)
os.makedirs('Other', exist_ok=True)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {DEVICE}")

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             f1_score, roc_auc_score,
                             precision_recall_curve)

In [ ]:
urlDB  = "https://raw.githubusercontent.com/FedericoSabbadini/f1-strategy-predictor/main/02_dbAnalysis/output/f1_dataset_clean.pkl"
urlENC = "https://raw.githubusercontent.com/FedericoSabbadini/f1-strategy-predictor/main/02_dbAnalysis/output/label_encoder.pkl"

# 2. Load Dataset

In [ ]:
def download_bytes(url):
    r = requests.get(url)
    r.raise_for_status()
    return r.content

df_f1         = pd.read_pickle(BytesIO(download_bytes(urlDB)))
label_encoder = joblib.load(BytesIO(download_bytes(urlENC)))

print(f"Dataset loaded   : {len(df_f1):,} rows, {df_f1.shape[1]} columns")
print(f"Compound classes : {list(label_encoder.classes_)}")

# 3. Define Features

In [ ]:
TARGETS   = ['PitIn3Laps', 'NextCompoundEnc']
META_COLS = ['Driver', 'Team', 'Circuit', 'Time', 'RaceName', 'Compound']

FEATURES = [col for col in df_f1.columns if col not in TARGETS + META_COLS]
n_feat   = len(FEATURES)

print(f"Number of features: {n_feat}")
print(f"Features: {FEATURES}")

# 4. Data Preprocessing

### 4.1 Create Race Sequences for Transformer

Una sequenza per pilota per gara (intera gara). La causal safety è garantita dalla
maschera causale lower-triangolare interna al Transformer, non troncando l'input.

In [ ]:
MAX_SEQ_LEN = int(df_f1.groupby(['Year', 'Round', 'Driver'])['LapNumber'].max().max())
print(f"Maximum sequence length: {MAX_SEQ_LEN}")

def create_race_sequences(df, target_col, features=FEATURES):
    """
    One sequence per driver per race.
    X[i] : (race_len, n_features)
    y[i] : (race_len,)
    """
    X, y = [], []
    for (_, _, _), group in df.groupby(['Year', 'Round', 'Driver']):
        group = group.sort_values('LapNumber').reset_index(drop=True)
        X.append(group[features].values.astype(np.float32))
        y.append(group[target_col].values)
    return X, y

### 4.2 Temporal Train / Val / Test Split

Split per RaceID in ordine cronologico: 70% train / 15% val / 15% test.

In [ ]:
df_clean = df_f1.sort_values(['Year', 'Round', 'LapNumber']).reset_index(drop=True)
df_clean['RaceID'] = df_clean['Year'].astype(str) + '_' + df_clean['Round'].astype(str).str.zfill(2)

SPLIT_PERC = [0.70, 0.85]
races = sorted(df_clean['RaceID'].unique())
n     = len(races)

df_train = df_clean[df_clean['RaceID'].isin(races[:int(SPLIT_PERC[0]*n)])].copy()
df_val   = df_clean[df_clean['RaceID'].isin(races[int(SPLIT_PERC[0]*n):int(SPLIT_PERC[1]*n)])].copy()
df_test  = df_clean[df_clean['RaceID'].isin(races[int(SPLIT_PERC[1]*n):])].copy()

print(f'Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')

In [ ]:
X_pit_train_raw,  y_pit_train_raw  = create_race_sequences(df_train, 'PitIn3Laps')
X_pit_val_raw,    y_pit_val_raw    = create_race_sequences(df_val,   'PitIn3Laps')
X_pit_test_raw,   y_pit_test_raw   = create_race_sequences(df_test,  'PitIn3Laps')

X_comp_train_raw, y_comp_train_raw = create_race_sequences(df_train, 'NextCompoundEnc')
X_comp_val_raw,   y_comp_val_raw   = create_race_sequences(df_val,   'NextCompoundEnc')
X_comp_test_raw,  y_comp_test_raw  = create_race_sequences(df_test,  'NextCompoundEnc')

lengths = [len(x) for x in X_pit_train_raw]
print(f"Races in train set: {len(X_pit_train_raw):,}")
print(f"Race length — Min: {min(lengths)} | Max: {max(lengths)} | Mean: {np.mean(lengths):.1f}")

### 4.3 Normalizzazione (RobustScaler)

In [ ]:
scaler_pit  = RobustScaler()
scaler_comp = RobustScaler()

def fit_scaler(sequences, scaler):
    scaler.fit(np.vstack(sequences))

def scale_sequences(sequences, scaler):
    scaled = []
    for s in sequences:
        t = scaler.transform(s)
        t[~np.isfinite(t)] = 0.0
        scaled.append(t.astype(np.float32))
    return scaled

fit_scaler(X_pit_train_raw,  scaler_pit)
fit_scaler(X_comp_train_raw, scaler_comp)

X_pit_train_raw  = scale_sequences(X_pit_train_raw,  scaler_pit)
X_pit_val_raw    = scale_sequences(X_pit_val_raw,    scaler_pit)
X_pit_test_raw   = scale_sequences(X_pit_test_raw,   scaler_pit)

X_comp_train_raw = scale_sequences(X_comp_train_raw, scaler_comp)
X_comp_val_raw   = scale_sequences(X_comp_val_raw,   scaler_comp)
X_comp_test_raw  = scale_sequences(X_comp_test_raw,  scaler_comp)

### 4.4 Padding

- **X** paddato con 0.0 (post-padding)
- **y** paddato con −1 (valore sentinella usato dalla masked loss)

In [ ]:
def pad_race_sequences(X_list, y_list, max_len, n_features=n_feat):
    N    = len(X_list)
    X    = np.zeros((N, max_len, n_features), dtype=np.float32)
    y    = np.full( (N, max_len), -1,         dtype=np.int64)
    mask = np.zeros((N, max_len),             dtype=bool)

    for i, (x_seq, y_seq) in enumerate(zip(X_list, y_list)):
        sLen = len(x_seq)
        X[i, :sLen]    = x_seq
        y[i, :sLen]    = y_seq
        mask[i, :sLen] = True

    return X, y, mask

X_pit_train,  y_pit_train,  mask_pit_train  = pad_race_sequences(X_pit_train_raw,  y_pit_train_raw,  MAX_SEQ_LEN)
X_pit_val,    y_pit_val,    mask_pit_val    = pad_race_sequences(X_pit_val_raw,    y_pit_val_raw,    MAX_SEQ_LEN)
X_pit_test,   y_pit_test,   mask_pit_test   = pad_race_sequences(X_pit_test_raw,   y_pit_test_raw,   MAX_SEQ_LEN)

X_comp_train, y_comp_train, mask_comp_train = pad_race_sequences(X_comp_train_raw, y_comp_train_raw, MAX_SEQ_LEN)
X_comp_val,   y_comp_val,   mask_comp_val   = pad_race_sequences(X_comp_val_raw,   y_comp_val_raw,   MAX_SEQ_LEN)
X_comp_test,  y_comp_test,  mask_comp_test  = pad_race_sequences(X_comp_test_raw,  y_comp_test_raw,  MAX_SEQ_LEN)

### 4.5 PyTorch Dataset & DataLoader

In [ ]:
class RaceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

loader_pit_train  = DataLoader(RaceDataset(X_pit_train,  y_pit_train),  batch_size=BATCH_SIZE, shuffle=True)
loader_pit_val    = DataLoader(RaceDataset(X_pit_val,    y_pit_val),    batch_size=BATCH_SIZE, shuffle=False)
loader_pit_test   = DataLoader(RaceDataset(X_pit_test,   y_pit_test),   batch_size=BATCH_SIZE, shuffle=False)

loader_comp_train = DataLoader(RaceDataset(X_comp_train, y_comp_train), batch_size=BATCH_SIZE, shuffle=True)
loader_comp_val   = DataLoader(RaceDataset(X_comp_val,   y_comp_val),   batch_size=BATCH_SIZE, shuffle=False)
loader_comp_test  = DataLoader(RaceDataset(X_comp_test,  y_comp_test),  batch_size=BATCH_SIZE, shuffle=False)

# 5. Architecture

## 5.1 Sinusoidal Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """
    Vaswani et al. 2017 — aggiunge encoding posizionale sinusoidale.
    Identico alla versione Keras, riscritta in PyTorch.
    """
    def __init__(self, max_len=200):
        super().__init__()
        self.max_len = max_len

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        seq_len, d_model = x.size(1), x.size(2)

        positions = torch.arange(self.max_len, dtype=torch.float32).unsqueeze(1)  # (max_len, 1)
        dims      = torch.arange(d_model,      dtype=torch.float32).unsqueeze(0)  # (1, d_model)

        angles  = positions / torch.pow(10000.0, (2 * (dims // 2)) / d_model)    # (max_len, d_model)
        pe      = torch.zeros(self.max_len, d_model)
        pe[:, 0::2] = torch.sin(angles[:, 0::2])
        pe[:, 1::2] = torch.cos(angles[:, 1::2])

        pe = pe[:seq_len].unsqueeze(0).to(x.device)  # (1, seq_len, d_model)
        return x + pe

## 5.2 Transformer Block (Pre-LN, Causal)

In [ ]:
class TransformerBlock(nn.Module):
    """
    Pre-LN Transformer block con causal masking.
    La maschera lower-triangolare sostituisce il troncamento dell'input:
    la posizione i può attenzionare solo le posizioni 0..i.
    Identico alla versione Keras, riscritta in PyTorch.
    """
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0

        self.attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.norm1  = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2  = nn.LayerNorm(d_model, eps=1e-6)
        self.drop1  = nn.Dropout(dropout)
        self.drop2  = nn.Dropout(dropout)

        # L2 sui pesi FFN (equivalente a kernel_regularizer=l2(1e-4) di Keras)
        for layer in self.ffn:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)

    def forward(self, x):
        seq_len = x.size(1)

        # Causal mask: additive, -inf sulle posizioni future
        # In Keras era una bool mask; PyTorch MultiheadAttention vuole una additive mask float
        causal_mask = torch.full((seq_len, seq_len), float('-inf'), device=x.device)
        causal_mask = torch.tril(causal_mask, diagonal=0)  # lower-triangular
        causal_mask = torch.tril(torch.zeros(seq_len, seq_len, device=x.device)).masked_fill(
            torch.tril(torch.ones(seq_len, seq_len, device=x.device), diagonal=0) == 0,
            float('-inf')
        )

        # Pre-LN: norma prima del sub-layer
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm, attn_mask=causal_mask)
        x = x + self.drop1(attn_out)

        x_norm = self.norm2(x)
        x = x + self.drop2(self.ffn(x_norm))

        return x

## 5.3 Masked Losses

y è paddato con −1. Le loss mascherano quelle posizioni.

In [ ]:
def masked_binary_crossentropy(y_pred, y_true):
    """
    y_true : (batch, seq_len)   — 0/1, -1 dove paddato
    y_pred : (batch, seq_len)   — logit (NON sigmoid, BCEWithLogitsLoss)
    """
    mask   = (y_true != -1).float()
    y_safe = torch.where(y_true == -1, torch.zeros_like(y_true), y_true).float()

    bce = nn.functional.binary_cross_entropy_with_logits(
        y_pred, y_safe, reduction='none'
    )  # (batch, seq_len)

    return (bce * mask).sum() / (mask.sum() + 1e-8)


def masked_categorical_crossentropy(y_pred, y_true):
    """
    y_true : (batch, seq_len)              — class indices, -1 dove paddato
    y_pred : (batch, seq_len, n_classes)   — logit (NON softmax)
    """
    mask   = (y_true != -1).float()
    y_safe = torch.where(y_true == -1, torch.zeros_like(y_true), y_true)  # evita indici -1

    # cross_entropy vuole (N, C, ...) come pred e (N, ...) come target
    sce = nn.functional.cross_entropy(
        y_pred.permute(0, 2, 1),   # (batch, n_classes, seq_len)
        y_safe,                    # (batch, seq_len)
        reduction='none'
    )  # (batch, seq_len)

    return (sce * mask).sum() / (mask.sum() + 1e-8)

## 5.4 Models

In [ ]:
class PitTransformer(nn.Module):
    """
    Transformer causale seq-to-seq per predire P(pit nei prossimi 3 giri).
    Output: (batch, seq_len) — logit per BCEWithLogitsLoss.
    """
    def __init__(self, n_features, max_len,
                 d_model=96, num_heads=4, ff_dim=288, num_blocks=3, dropout=0.4):
        super().__init__()

        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = SinusoidalPositionalEncoding(max_len=max_len)
        self.input_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_blocks)
        ])

        self.pre_head_norm = nn.LayerNorm(d_model, eps=1e-6)
        self.head_dense    = nn.Linear(d_model, 32)
        self.head_act      = nn.GELU()
        self.head_drop     = nn.Dropout(dropout)
        self.output        = nn.Linear(32, 1)

    def forward(self, x):
        x = self.input_proj(x)    # (batch, seq, d_model)
        x = self.pos_enc(x)
        x = self.input_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.pre_head_norm(x)
        x = self.head_act(self.head_dense(x))
        x = self.head_drop(x)
        x = self.output(x).squeeze(-1)  # (batch, seq_len)
        return x


model_pit = PitTransformer(n_features=n_feat, max_len=MAX_SEQ_LEN).to(DEVICE)
print(f"PIT model params: {sum(p.numel() for p in model_pit.parameters()):,}")

In [ ]:
class CompoundTransformer(nn.Module):
    """
    Transformer causale seq-to-seq per predire il prossimo compound.
    Output: (batch, seq_len, n_classes) — logit per CrossEntropyLoss.
    """
    def __init__(self, n_features, max_len, n_classes,
                 d_model=128, num_heads=4, ff_dim=384, num_blocks=4, dropout=0.4):
        super().__init__()

        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = SinusoidalPositionalEncoding(max_len=max_len)
        self.input_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_blocks)
        ])

        self.pre_head_norm = nn.LayerNorm(d_model, eps=1e-6)
        self.head_dense1   = nn.Linear(d_model, 48)
        self.head_act1     = nn.GELU()
        self.head_drop1    = nn.Dropout(dropout)
        self.head_dense2   = nn.Linear(48, 24)
        self.head_act2     = nn.GELU()
        self.head_drop2    = nn.Dropout(dropout * 0.5)
        self.output        = nn.Linear(24, n_classes)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.input_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.pre_head_norm(x)
        x = self.head_act1(self.head_dense1(x))
        x = self.head_drop1(x)
        x = self.head_act2(self.head_dense2(x))
        x = self.head_drop2(x)
        x = self.output(x)          # (batch, seq_len, n_classes)
        return x


n_classes  = len(label_encoder.classes_)
model_comp = CompoundTransformer(n_features=n_feat, max_len=MAX_SEQ_LEN, n_classes=n_classes).to(DEVICE)
print(f"COMP model params: {sum(p.numel() for p in model_comp.parameters()):,}")

# 6. Training

In [ ]:
def train_model(model, loss_fn, optimizer, scheduler,
                loader_train, loader_val,
                epochs=50, patience=20, mode='pit'):
    """
    Loop di training generico.
    mode='pit'  → monitora AUC su val (early stopping su max)
    mode='comp' → monitora accuracy su val (early stopping su max)
    """
    best_val_metric = -np.inf
    best_state      = None
    patience_cnt    = 0
    history         = {'train_loss': [], 'val_loss': [], 'val_metric': []}

    for epoch in range(1, epochs + 1):
        # --- TRAIN ---
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in loader_train:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            out  = model(X_batch)
            loss = loss_fn(out, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(loader_train)

        # --- VAL ---
        model.eval()
        val_loss   = 0.0
        all_probs, all_preds, all_trues = [], [], []

        with torch.no_grad():
            for X_batch, y_batch in loader_val:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                out  = model(X_batch)
                loss = loss_fn(out, y_batch)
                val_loss += loss.item()

                mask = (y_batch != -1).cpu().numpy().ravel()
                if mode == 'pit':
                    probs = torch.sigmoid(out).cpu().numpy().ravel()[mask]
                    all_probs.extend(probs)
                    all_trues.extend(y_batch.cpu().numpy().ravel()[mask])
                else:
                    preds = out.argmax(dim=-1).cpu().numpy().ravel()[mask]
                    trues = y_batch.cpu().numpy().ravel()[mask]
                    all_preds.extend(preds)
                    all_trues.extend(trues)

        val_loss /= len(loader_val)

        if mode == 'pit':
            val_metric = roc_auc_score(all_trues, all_probs)
            metric_name = 'AUC'
        else:
            val_metric  = accuracy_score(all_trues, all_preds)
            metric_name = 'Acc'

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_metric'].append(val_metric)

        print(f"Epoch {epoch:3d}/{epochs} — train_loss: {train_loss:.4f}  "
              f"val_loss: {val_loss:.4f}  val_{metric_name}: {val_metric:.4f}  "
              f"lr: {optimizer.param_groups[0]['lr']:.2e}")

        if val_metric > best_val_metric:
            best_val_metric = val_metric
            best_state      = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt    = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"Early stopping at epoch {epoch} (patience={patience})")
                break

    model.load_state_dict(best_state)  # restore best weights
    print(f"Best val {metric_name}: {best_val_metric:.4f}")
    return history

In [ ]:
optimizer_pit = optim.AdamW(model_pit.parameters(),
                             lr=0.003326, weight_decay=2.762e-5)
scheduler_pit = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_pit, mode='min', factor=0.5, patience=8, verbose=True
)

print("Training PIT Transformer (seq-to-seq)...")
print("=" * 60)

history_pit = train_model(
    model_pit, masked_binary_crossentropy,
    optimizer_pit, scheduler_pit,
    loader_pit_train, loader_pit_val,
    epochs=50, patience=20, mode='pit'
)

In [ ]:
optimizer_comp = optim.AdamW(model_comp.parameters(),
                              lr=0.00019895, weight_decay=0.0001983)
scheduler_comp = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_comp, mode='min', factor=0.6, patience=8, verbose=True
)

print("\nTraining COMPOUND Transformer (seq-to-seq)...")
print("=" * 60)

history_comp = train_model(
    model_comp, masked_categorical_crossentropy,
    optimizer_comp, scheduler_comp,
    loader_comp_train, loader_comp_val,
    epochs=50, patience=18, mode='comp'
)

# 7. Evaluation

In [ ]:
print("-" * 50)
print("PIT MODEL (PitIn3Laps)")
print("-" * 50)

model_pit.eval()
pit_pred_prob_list, pit_true_list = [], []

with torch.no_grad():
    for X_batch, y_batch in loader_pit_test:
        out  = model_pit(X_batch.to(DEVICE)).cpu()
        prob = torch.sigmoid(out).numpy()
        mask = (y_batch.numpy() != -1).ravel()
        pit_pred_prob_list.extend(prob.ravel()[mask])
        pit_true_list.extend(y_batch.numpy().ravel()[mask])

pit_pred_prob = np.array(pit_pred_prob_list)
pit_true      = np.array(pit_true_list)

prec, rec, thresh = precision_recall_curve(pit_true, pit_pred_prob)
f1_arr     = 2 * prec * rec / (prec + rec + 1e-8)
best_idx   = np.argmax(f1_arr)
opt_thresh = thresh[best_idx] if best_idx < len(thresh) else 0.5

pit_pred = (pit_pred_prob > opt_thresh).astype(int)

pit_auc = roc_auc_score(pit_true, pit_pred_prob)
pit_f1  = f1_score(pit_true, pit_pred)
pit_acc = accuracy_score(pit_true, pit_pred)

print(f"AUC-ROC:   {pit_auc:.3f}")
print(f"F1 Score:  {pit_f1:.3f}")
print(f"Accuracy:  {pit_acc:.3f}")
print(f"Precision: {prec[best_idx]:.3f}")
print(f"Recall:    {rec[best_idx]:.3f}")
print(f"Threshold: {opt_thresh:.3f}")

cm_pit = confusion_matrix(pit_true, pit_pred)
print(f"\nConfusion Matrix:")
print(f"           Pred:0  Pred:1")
print(f"True:0     {cm_pit[0,0]:6d}  {cm_pit[0,1]:6d}")
print(f"True:1     {cm_pit[1,0]:6d}  {cm_pit[1,1]:6d}")

In [ ]:
print("-" * 50)
print("COMPOUND MODEL (NextCompound)")
print("-" * 50)

model_comp.eval()
comp_pred_list, comp_true_list = [], []

with torch.no_grad():
    for X_batch, y_batch in loader_comp_test:
        out  = model_comp(X_batch.to(DEVICE)).cpu()
        pred = out.argmax(dim=-1).numpy()
        mask = (y_batch.numpy() != -1).ravel()
        comp_pred_list.extend(pred.ravel()[mask])
        comp_true_list.extend(y_batch.numpy().ravel()[mask])

comp_pred = np.array(comp_pred_list)
comp_true = np.array(comp_true_list)

comp_acc = accuracy_score(comp_true, comp_pred)
comp_f1  = f1_score(comp_true, comp_pred, average='weighted')

print(f"Accuracy:      {comp_acc:.3f}")
print(f"F1 (weighted): {comp_f1:.3f}")

print("\nPer-class accuracy:")
for i, cls in enumerate(label_encoder.classes_):
    mask_cls       = comp_true == i
    cls_acc        = (comp_pred[mask_cls] == i).mean() if mask_cls.sum() > 0 else 0.0
    cls_pred_count = (comp_pred == i).sum()
    print(f"  {cls:12s}: {cls_acc:.3f}  (n={mask_cls.sum():,}, pred={cls_pred_count:,})")

cm_comp = confusion_matrix(comp_true, comp_pred)
print(f"\nConfusion Matrix (classes: {list(label_encoder.classes_)}):")
print(cm_comp)

# 8. Training History Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Transformer Seq-to-Seq (PyTorch) — Training History', fontsize=14, fontweight='bold')

axes[0, 0].plot(history_pit['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history_pit['val_loss'],   label='Validation', linewidth=2)
axes[0, 0].set_title('PIT Model — Loss')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history_pit['val_metric'], label='Val AUC', linewidth=2)
axes[0, 1].axhline(y=pit_auc, color='r', linestyle='--', label=f'Test AUC: {pit_auc:.3f}')
axes[0, 1].set_title('PIT Model — AUC')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim(0.5, 1.0)

axes[1, 0].plot(history_comp['train_loss'], label='Train', linewidth=2)
axes[1, 0].plot(history_comp['val_loss'],   label='Validation', linewidth=2)
axes[1, 0].set_title('COMPOUND Model — Loss')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history_comp['val_metric'], label='Val Accuracy', linewidth=2)
axes[1, 1].axhline(y=comp_acc, color='r', linestyle='--', label=f'Test Acc: {comp_acc:.3f}')
axes[1, 1].set_title('COMPOUND Model — Accuracy')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim(0.2, 1.0)

plt.tight_layout()
plt.savefig('Other/transformer_seq2seq_history.png', dpi=150, bbox_inches='tight')
plt.show()

# 9. Save

In [ ]:
torch.save(model_pit.state_dict(),  'Model/f1_pit_transformer_seq2seq.pt')
torch.save(model_comp.state_dict(), 'Model/f1_compound_transformer_seq2seq.pt')
print("Models saved.")

In [ ]:
joblib.dump(scaler_pit,    'Model/f1_pit_scaler.pkl')
joblib.dump(scaler_comp,   'Model/f1_comp_scaler.pkl')
joblib.dump(label_encoder, 'Model/label_encoder.pkl')
print("Scalers saved.")

In [ ]:
config = {
    'architecture'     : 'causal_transformer_seq2seq_pytorch',
    'sequence_strategy': 'one_sequence_per_race',
    'max_seq_len'      : MAX_SEQ_LEN,
    'n_features'       : n_feat,
    'pit_model': {
        'd_model': 96, 'num_heads': 4, 'ff_dim': 288, 'num_blocks': 3
    },
    'comp_model': {
        'd_model': 128, 'num_heads': 4, 'ff_dim': 384, 'num_blocks': 4
    },
    'FEATURES'         : FEATURES,
    'pit_threshold'    : float(opt_thresh),
    'compound_classes' : list(label_encoder.classes_),
    'metrics': {
        'pit_auc'          : float(pit_auc),
        'pit_f1'           : float(pit_f1),
        'pit_accuracy'     : float(pit_acc),
        'compound_accuracy': float(comp_acc),
        'compound_f1'      : float(comp_f1)
    }
}

with open('Model/transformerSeq2SeqConfig.json', 'w') as f:
    json.dump(config, f, indent=2)

df_f1.to_pickle('Model/f1_dataset_featured.pkl')
print("Config and dataset saved.")